# Context Strategy Eval — Gemini Answer Model

Uses Gemini 2.5 Flash via Vertex AI for both answering and summarization. No local GPU model required.

## 1. Authenticate with Google

In [ ]:
from google.colab import auth
auth.authenticate_user()

In [ ]:
from huggingface_hub import login
login()

## 2. HuggingFace Login\n\nRequired for downloading the LLMLingua compression model. Token needs no special permissions — read-only is fine.

## 2. Clone Repo

In [ ]:
from pathlib import Path
import os

REPO_URL = 'https://github.com/rissingh23/Context-Opt.git'
REPO_DIR = Path('/content/Context-Opt')
BRANCH = 'gemini-eval'

if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
!git checkout {BRANCH}
print('Working directory:', Path.cwd())

## 3. Install Dependencies

In [ ]:
!pip install -r requirements.txt

## 4. Config

Set your GCP project ID here.

In [ ]:
GCP_PROJECT = 'contextops-498110'
GCP_LOCATION = 'us-central1'
GEMINI_MODEL = 'gemini-2.5-flash'

# Quick connectivity test
from google import genai
client = genai.Client(vertexai=True, project=GCP_PROJECT, location=GCP_LOCATION)
response = client.models.generate_content(model=GEMINI_MODEL, contents='Reply with OK.')
print('Gemini connection:', response.text)

## 5. Download LongBench Data

In [ ]:
!python3 -m src.data.download_longbench \
  --tasks qasper hotpotqa gov_report multi_news passage_count \
  --limit 10

## 6. Smoke Test — 1 Example, All Strategies, Real Gemini Answers

This is the first real eval run — quality scores will reflect actual model performance.

In [ ]:
!python3 -m src.eval_framework.run_eval_table \
  --tasks qasper \
  --strategies full_context retrieval compression gemini_summarization retrieval_compression \
  --limit 1 \
  --provider vertexai \
  --model gemini-2.5-flash \
  --vertexai-project {GCP_PROJECT} \
  --vertexai-location {GCP_LOCATION} \
  --rows-output outputs/processed/gemini_smoke_rows.csv \
  --aggregate-output outputs/processed/gemini_smoke_summary.csv \
  --json-output outputs/processed/gemini_smoke_rows.jsonl

In [ ]:
import pandas as pd
pd.read_csv('outputs/processed/gemini_smoke_summary.csv')

## 7. Full Eval — All Tasks, 10 Examples Each

In [ ]:
!python3 -m src.eval_framework.run_eval_table \
  --tasks qasper hotpotqa gov_report multi_news passage_count \
  --strategies full_context retrieval compression gemini_summarization retrieval_compression \
  --limit 10 \
  --provider vertexai \
  --model gemini-2.5-flash \
  --vertexai-project {GCP_PROJECT} \
  --vertexai-location {GCP_LOCATION} \
  --rows-output outputs/processed/gemini_eval_rows.csv \
  --aggregate-output outputs/processed/gemini_eval_summary.csv \
  --json-output outputs/processed/gemini_eval_rows.jsonl

In [ ]:
import pandas as pd
df = pd.read_csv('outputs/processed/gemini_eval_summary.csv')
df[['task', 'strategy', 'num_examples', 'error_rate', 'avg_quality_score',
    'avg_rouge_l', 'avg_token_f1', 'avg_compression_ratio',
    'avg_total_latency_sec', 'avg_estimated_cost']]

## 8. Inspect Errors

In [ ]:
rows = pd.read_csv('outputs/processed/gemini_eval_rows.csv')
errors = rows[rows['error'].fillna('') != '']
print('Error rows:', len(errors))
if len(errors):
    print(errors[['task', 'strategy', 'error']].to_string())